## Ejercico 01

Objetivo: En este ejercicio vamos a implementar una búsqueda dentro de un corpus  de 100 textos

## _Pasos:_
1. Leer el corpus

In [1]:
import os 

# Usamos 'r' antes de la cadena para evitar problemas con las barras invertidas en Windows
path = r"C:\Users\ASUS\Documents\7mo Semestre\RI\1erLab\ir26a\data\gutenberg\libros_gutenberg"

contenidos = {}
for archivo in os.listdir(path):
    ruta_completa = os.path.join(path, archivo)
    try:
        with open(ruta_completa, "r", encoding="utf-8") as f:
            contenidos[archivo] = f.read()
    except UnicodeDecodeError:
        # Respaldo por si algún libro tiene otra codificación
        with open(ruta_completa, "r", encoding="latin-1") as f:
            contenidos[archivo] = f.read()

# Imprimimos el resultado una sola vez, fuera del bucle
print(f"\n¡Listo! Se encontraron y leyeron {len(contenidos)} archivos.\n")


¡Listo! Se encontraron y leyeron 94 archivos.



2._ Busqueda en 100 libros

In [2]:
import time

start_time = time.time()
query = "Mary"
resultados = []
for key in contenidos.keys():
    if query in contenidos[key]:
        resultados.append(key)

print(f"Los archivos que contienen la palabra '{query}' son: {resultados}")
end_time = time.time()
print(f"Tiempo de búsqueda: {end_time - start_time} segundos")
print(f"Se encontraron {len(resultados)} archivos que contienen la palabra '{query}'")

Los archivos que contienen la palabra 'Mary' son: ['ebook_100.txt', 'ebook_11.txt', 'ebook_1184.txt', 'ebook_1232.txt', 'ebook_1260.txt', 'ebook_13188.txt', 'ebook_1342.txt', 'ebook_1400.txt', 'ebook_145.txt', 'ebook_161.txt', 'ebook_16328.txt', 'ebook_1661.txt', 'ebook_19488.txt', 'ebook_1952.txt', 'ebook_20203.txt', 'ebook_205.txt', 'ebook_2160.txt', 'ebook_23.txt', 'ebook_2440.txt', 'ebook_2465.txt', 'ebook_2554.txt', 'ebook_2600.txt', 'ebook_2641.txt', 'ebook_2701.txt', 'ebook_28054.txt', 'ebook_3011.txt', 'ebook_31885.txt', 'ebook_3207.txt', 'ebook_3268.txt', 'ebook_3296.txt', 'ebook_34413.txt', 'ebook_345.txt', 'ebook_36034.txt', 'ebook_37106.txt', 'ebook_37683.txt', 'ebook_394.txt', 'ebook_4085.txt', 'ebook_4300.txt', 'ebook_45.txt', 'ebook_47715.txt', 'ebook_49008.txt', 'ebook_50559.txt', 'ebook_51252.txt', 'ebook_51960.txt', 'ebook_5197.txt', 'ebook_52106.txt', 'ebook_57336.txt', 'ebook_58031.txt', 'ebook_59131.txt', 'ebook_59231.txt', 'ebook_60230.txt', 'ebook_65238.txt', 'eb

Paso 2: Tokenización y Creación del Índice Invertido

Esta celda reemplaza la necesidad de buscar texto por texto. Va a extraer todas las palabras únicas de cada libro, las convertirá a minúsculas (para evitar duplicados como "Mary" y "mary"), y creará nuestro "diccionario" de búsqueda.

In [3]:
import re
from collections import defaultdict
import time

print("Construyendo el índice invertido para los 100 libros...")
start_time = time.time()

# Usamos defaultdict porque crea un conjunto (set) automáticamente si la palabra no existe
indice_invertido = defaultdict(set)
palabras_corpus = set() # Guardaremos todas las palabras únicas para la prueba final

for archivo, texto in contenidos.items():
    # Expresión regular para extraer solo palabras, ignorando puntos, comas, etc.
    palabras = set(re.findall(r'\b\w+\b', texto.lower()))
    
    # Agregamos las palabras de este libro al gran total
    palabras_corpus.update(palabras)
    
    # Poblamos el índice invertido
    for palabra in palabras:
        indice_invertido[palabra].add(archivo)

end_time = time.time()
print(f"Índice construido en: {end_time - start_time:.4f} segundos")
print(f"Total de palabras únicas en el corpus: {len(indice_invertido)}")

Construyendo el índice invertido para los 100 libros...
Índice construido en: 5.8149 segundos
Total de palabras únicas en el corpus: 250400


Paso 3: Función de Búsqueda Optimizada
Esta celda reemplaza tu función buscar(query, contenidos). Ahora la búsqueda es de complejidad $O(1)$ porque consulta directamente la llave en el diccionario.

In [4]:
def buscar_en_indice(query, indice):
    # Aseguramos que la consulta tenga el mismo formato que nuestro índice (minúsculas)
    query_limpia = query.lower().strip()
    
    # .get() devuelve los archivos si la palabra existe, o una lista vacía [] si no
    return list(indice.get(query_limpia, []))

# Replicando tu celda de prueba con "Mary"
start_time = time.time()
query = "Mary"
resultados = buscar_en_indice(query, indice_invertido)
end_time = time.time()

print(f"La palabra '{query}' aparece en {len(resultados)} libros.")
print(f"Lista de archivos: {resultados[:10]}...") # Mostramos solo los primeros 10 para no saturar la pantalla
print(f"Tiempo de búsqueda: {end_time - start_time:.6f} segundos")

La palabra 'Mary' aparece en 65 libros.
Lista de archivos: ['ebook_74.txt', 'ebook_161.txt', 'ebook_1661.txt', 'ebook_19488.txt', 'ebook_36034.txt', 'ebook_2465.txt', 'ebook_345.txt', 'ebook_34413.txt', 'ebook_59131.txt', 'ebook_25344.txt']...
Tiempo de búsqueda: 0.000000 segundos


## Paso 4: Prueba de Estrés 
En el primer  archivo base, buscar ~12,000 palabras en unos pocos libros tomó más de 200 segundos. Aquí vamos a buscar todas las palabras únicas de los 100 libros para demostrar la eficiencia del índice.

In [5]:
print(f"Iniciando búsqueda masiva de {len(palabras_corpus)} palabras en el índice...")
start_time = time.time()

# Simulamos la búsqueda de cada una de las miles de palabras en el corpus
for p in palabras_corpus:
    buscar_en_indice(p, indice_invertido)

end_time = time.time()

# Este tiempo debería ser increíblemente más bajo, probablemente menos de 1 segundo.
print(f"Tiempo total para realizar {len(palabras_corpus)} búsquedas: {(end_time - start_time):.6f} segundos")

Iniciando búsqueda masiva de 250400 palabras en el índice...
Tiempo total para realizar 250400 búsquedas: 0.165510 segundos
